# Lab 07: Feedback (Closing the Loop)
Formatting failures as JSON 'EDA Failure Cards' and feeding them back.


In [ ]:
import json
import requests

with open('.arch2_state.json', 'r') as f: state = json.load(f)
cycles = state.get('proxy_cycles', 999999)
max_cycles = state.get('max_cycles', 500000)

if cycles > max_cycles:
    print(f"FAIL: Design took {cycles} cycles (Max: {max_cycles}).")
    failure_card = {
        "error": "TimingViolation",
        "details": f"cycles={cycles} > max={max_cycles}",
        "action": "Increase ArrayHeight and ArrayWidth significantly."
    }
    print("EDA Failure Card:\n", json.dumps(failure_card, indent=2))
    
    # Reprompt LLM
    prompt = f"Previous config failed. Failure Card: {json.dumps(failure_card)}. Propose a much larger systolic array config. Output ONLY JSON."
    res = requests.post("http://localhost:11434/api/generate", json={"model": "gemma3:1b", "prompt": prompt, "stream": False})
    raw_resp = res.json()['response']
    print("LLM Feedback Correction:\n", raw_resp)
    
    import re
    match = re.search(r'\{.*\}', raw_resp.replace('\n', ''))
    if match:
        try:
            state['ai_config'] = json.loads(match.group(0))
        except:
            state['ai_config'] = {"ArrayHeight": 32, "ArrayWidth": 32}
    else:
        state['ai_config'] = {"ArrayHeight": 32, "ArrayWidth": 32}
else:
    print("PASS: Design met constraints!")

with open('.arch2_state.json', 'w') as f: json.dump(state, f)
